<div dir="rtl" align="right">
# تعزيزُ البياناتِ لإشاراتِ EEG

## نظرةٌ عامّةٌ
يُطبّقُ هذا الدفترُ ثلاثَ تقنياتٍ لِتعزيزِ البياناتِ على إشاراتِ EEG ويُقارنُ أداءَ EEGNet بِوجودِ التعزيزِ وبدونَه.

**التقنيات:** الإزاحةُ الزمنيّةُ، إسقاطُ القنواتِ، الضوضاءُ الغاوسيّةُ

## المُعاملاتُ الأساسيةُ
| المعامل | القيمة | المعنى |
|---------|--------|---------|
| max_shift | 50 | أقصى إزاحة زمنيّة |
| p_drop | 0.2 | احتماليّة إسقاط القناة |
| sigma | 0.1 | انحراف الضوضاء |
</div>

<div dir="rtl" align="right">
## 1. تثبيتُ المكتباتِ
</div>

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn torch

<div dir="rtl" align="right">
## 2. تحميلُ البياناتِ
</div>

In [ ]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

y = np.array([0 if lab == 'left_hand' else 1 for lab in labels])
print(f"Data shape: {X.shape}")
print(f"Labels: {len(labels)}")
print(f"Classes: {np.unique(labels)}")

<div dir="rtl" align="right">
## 3. تعريفُ دوالِ التعزيزِ
</div>

In [ ]:
def time_shift(signal, max_shift=50):
    shifted = np.zeros_like(signal)
    shift = np.random.randint(-max_shift, max_shift + 1)
    if shift == 0:
        return signal.copy()
    if shift > 0:
        shifted[:, shift:] = signal[:, :-shift]
    else:
        shifted[:, :shift] = signal[:, -shift:]
    return shifted

def channel_dropout(signal, p=0.2):
    dropped = signal.copy()
    drop_mask = np.random.rand(signal.shape[0]) < p
    dropped[drop_mask] = 0.0
    return dropped

def gaussian_noise(signal, sigma=0.1):
    return signal + np.random.normal(0, sigma, signal.shape)

np.random.seed(0)
example = X[0]
ex_shift = time_shift(example, max_shift=50)
ex_dropout = channel_dropout(example, p=0.2)
ex_noise = gaussian_noise(example, sigma=0.1)

<div dir="rtl" align="right">
## 4. تدريبُ النموذجِ بِوجودِ التعزيزِ وبدونَه
</div>

In [ ]:
import torch
import torch.nn as nn

class EEGNet(nn.Module):
    def __init__(self, n_channels=22, n_samples=1001, n_classes=2, F1=8, D=2, F2=16, dropout=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, F1, (1, n_samples // 2), padding='same')
        self.batchnorm1 = nn.BatchNorm2d(F1)
        self.depthwise = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1)
        self.batchnorm2 = nn.BatchNorm2d(F1 * D)
        self.activation = nn.ELU()
        self.pool1 = nn.AvgPool2d((1, 4))
        self.dropout1 = nn.Dropout(dropout)
        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding='same'),
            nn.Conv2d(F1 * D, F2, (1, 1)),
        )
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.dropout2 = nn.Dropout(dropout)
        dummy = torch.zeros(1, 1, n_channels, n_samples)
        out = self._features(dummy)
        self.classify = nn.Linear(out.view(-1).shape[0], n_classes)

    def _features(self, x):
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.depthwise(x)
        x = self.batchnorm2(x)
        x = self.activation(x)
        x = self.pool1(x)
        x = self.dropout1(x)
        x = self.separable(x)
        x = self.batchnorm3(x)
        x = self.activation(x)
        x = self.pool2(x)
        x = self.dropout2(x)
        return x

    def forward(self, x):
        x = self._features(x)
        x = x.view(x.size(0), -1)
        x = self.classify(x)
        return x

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
np.random.seed(42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

n_channels = X.shape[1]
n_samples = X.shape[2]

def augment_batch(X_batch):
    augmented = np.zeros_like(X_batch)
    for i in range(X_batch.shape[0]):
        choice = np.random.randint(0, 3)
        if choice == 0:
            augmented[i] = time_shift(X_batch[i])
        elif choice == 1:
            augmented[i] = channel_dropout(X_batch[i])
        else:
            augmented[i] = gaussian_noise(X_batch[i])
    return augmented

def train_and_eval(use_aug, epochs=30):
    torch.manual_seed(42)
    np.random.seed(42)
    model = EEGNet(n_channels=n_channels, n_samples=n_samples, n_classes=2)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        perm = torch.randperm(X_train_t.shape[0])
        for start in range(0, X_train_t.shape[0], 32):
            idx = perm[start:start + 32]
            batch_x = X_train_t[idx].numpy()
            batch_y = y_train_t[idx]
            if use_aug:
                batch_x = augment_batch(batch_x)
            batch_x_t = torch.tensor(batch_x, dtype=torch.float32).unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(batch_x_t), batch_y)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_t.unsqueeze(1))
        _, predicted = torch.max(outputs, 1)
    return accuracy_score(y_test_t.numpy(), predicted.numpy())

acc_plain = train_and_eval(use_aug=False)
print(f"Accuracy without augmentation: {acc_plain:.4f}")

torch.manual_seed(42)
np.random.seed(42)
acc_aug = train_and_eval(use_aug=True)
print(f"Accuracy with augmentation: {acc_aug:.4f}")

<div dir="rtl" align="right">
## 5. رسمٌ تفاعليٌّ
</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=3, subplot_titles=('Time Shift', 'Channel Dropout', 'Gaussian Noise', '', '', ''),
                    vertical_spacing=0.15)

fig.add_trace(go.Scatter(y=example[0], name='Original', opacity=0.7), row=1, col=1)
fig.add_trace(go.Scatter(y=ex_shift[0], name='Shifted', opacity=0.7), row=1, col=1)
fig.add_trace(go.Scatter(y=example[0], name='Original', opacity=0.7, showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=ex_dropout[0], name='Dropped', opacity=0.7), row=1, col=2)
fig.add_trace(go.Scatter(y=example[0], name='Original', opacity=0.7, showlegend=False), row=1, col=3)
fig.add_trace(go.Scatter(y=ex_noise[0], name='Noisy', opacity=0.7), row=1, col=3)

fig.add_trace(go.Bar(x=['No Augmentation', 'With Augmentation'], y=[acc_plain, acc_aug],
                     marker_color=['steelblue', 'coral'], text=[f'{acc_plain:.4f}', f'{acc_aug:.4f}'],
                     textposition='outside', showlegend=False), row=2, col=1)
fig.update_xaxes(row=2, col=1, domain=[0.1, 0.9])

fig.update_layout(title='Data Augmentation for EEG', width=1100, height=700)
fig.show()

<div dir="rtl" align="right">
## خلاصةٌ
- التعزيزُ يُزيدُ تنوّعَ بياناتِ التدريب
- ثلاثُ تقنياتٍ: إزاحةٌ زمنيّةٌ، إسقاطُ قنواتٍ، ضوضاءٌ غاوسيّةٌ
- التعزيزُ يُفيدُ أكثرَ حينَ تكونُ البياناتُ صغيرةً
</div>